# Урок 5. Табличные модели и матрица смежности

9 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 4](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-04.ipynb) · [Урок 6 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-06.ipynb)

---

Представление графа таблицей. Матрица смежности и весовая матрица. Переход от таблицы к рисунку и обратно.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-05", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Граф без рисунка

На прошлом уроке граф был картинкой. Но в задачах ОГЭ его чаще задают
**таблицей**: её нельзя понять двояко, и с ней умеет работать компьютер.

### Матрица смежности

> **Матрица смежности** — квадратная таблица, где строки и столбцы
> соответствуют вершинам, а на пересечении стоит 1, если ребро есть,
> и 0, если нет.

Для графа

```
       А ───── Б
       │       │
       В ───── Г
```

матрица выглядит так:

|  | А | Б | В | Г |
|---|---|---|---|---|
| **А** | 0 | 1 | 1 | 0 |
| **Б** | 1 | 0 | 0 | 1 |
| **В** | 1 | 0 | 0 | 1 |
| **Г** | 0 | 1 | 1 | 0 |

Три полезных наблюдения.

1. **На главной диагонали нули** — вершина не соединена сама с собой
   (если только в задаче нет петель).
2. **Матрица симметрична** относительно главной диагонали — для
   неориентированного графа. У орграфа симметрии не будет.
3. **Сумма чисел в строке равна степени вершины.**

### Весовая матрица

Если рёбра имеют вес — длину дороги, стоимость, время, — вместо единиц
записывают эти числа. Отсутствие связи обозначают прочерком, нулём
или пустой клеткой.

Именно так выглядят задачи ОГЭ про дороги:

|  | А | Б | В | Г | Д |
|---|---|---|---|---|---|
| **А** | — | 4 | 7 | — | — |
| **Б** | 4 | — | 2 | 9 | — |
| **В** | 7 | 2 | — | 3 | 6 |
| **Г** | — | 9 | 3 | — | 5 |
| **Д** | — | — | 6 | 5 | — |

### Как читать такую таблицу

Типичная формулировка: «В таблице отражена протяжённость дорог между
пунктами. Определите длину кратчайшего пути из А в Д». Порядок работы:

1. **Восстановите рисунок.** Нарисуйте кружки и подпишите рёбра весами —
   по картинке думать несравнимо легче, чем по таблице.
2. **Проверьте симметрию.** В школьных задачах дороги двусторонние,
   и таблица обязана быть симметричной. Несимметричность — сигнал,
   что вы неверно прочитали условие.
3. **Не считайте прочерк нулём.** Прочерк означает «дороги нет»,
   а ноль означал бы «дорога нулевой длины». Это принципиально разные вещи.

### Матрица смежности в коде

В программе матрицу удобно хранить списком списков:

```python
матрица = [
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0],
]
```

Обращение к элементу — `матрица[строка][столбец]`. Нумерация с нуля,
поэтому вершине А соответствует индекс 0, Б — 1 и так далее.

### Матрица или список смежности

Оба способа описывают одно и то же, но подходят для разных задач.

| | Матрица смежности | Список смежности |
|---|---|---|
| Проверить, есть ли ребро | мгновенно | нужен перебор соседей |
| Перечислить соседей | перебрать всю строку | сразу готов |
| Память при 1000 вершин | миллион ячеек | только реальные связи |

Вывод: для плотных графов (связей много) удобнее матрица,
для разреженных (связей мало) — список.

## Смотрим, как это работает

### Пример 1. Читаем матрицу

In [ ]:
имена = ["А", "Б", "В", "Г"]
матрица = [
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0],
]

# Печатаем с заголовками, чтобы было похоже на условие задачи
print("     " + "  ".join(имена))
for i, строка in enumerate(матрица):
    print(f"  {имена[i]}  " + "  ".join(str(з) for з in строка))

print()
for i, имя in enumerate(имена):
    соседи = [имена[j] for j in range(len(имена)) if матрица[i][j] == 1]
    print(f"  {имя}: степень {sum(матрица[i])}, соседи {', '.join(соседи)}")

Функция `enumerate` выдаёт сразу и номер, и элемент — это избавляет
от возни с индексами. А `sum(матрица[i])` считает степень вершины
одной операцией: единицы в строке как раз и есть рёбра.

### Пример 2. Из матрицы в список смежности

Переход между двумя представлениями — типовая операция.

In [ ]:
def в_список_смежности(матрица, имена):
    граф = {}
    for i, имя in enumerate(имена):
        граф[имя] = [имена[j] for j in range(len(имена)) if матрица[i][j] != 0]
    return граф


граф = в_список_смежности(матрица, имена)
for вершина, соседи in граф.items():
    print(f"  {вершина} → {соседи}")

Условие `!= 0` вместо `== 1` выбрано специально: так функция подойдёт
и для весовой матрицы, где вместо единиц стоят длины дорог.

### Пример 3. Задача ОГЭ на восстановление графа

> Между пунктами А–Д построены дороги. Определите, из какого пункта
> выходит больше всего дорог.

Разберём весовую матрицу.

In [ ]:
пункты = ["А", "Б", "В", "Г", "Д"]
# 0 означает, что дороги нет
веса = [
    [0, 4, 7, 0, 0],
    [4, 0, 2, 9, 0],
    [7, 2, 0, 3, 6],
    [0, 9, 3, 0, 5],
    [0, 0, 6, 5, 0],
]

print("Проверка симметричности:", веса == [list(строка) for строка in zip(*веса)])
print()

for i, пункт in enumerate(пункты):
    дорог = sum(1 for вес in веса[i] if вес != 0)
    длина = sum(веса[i])
    print(f"  {пункт}: дорог {дорог}, суммарная длина {длина}")

Строка `zip(*веса)` транспонирует матрицу — меняет строки и столбцы
местами. Если после транспонирования матрица не изменилась, значит она
симметрична, и граф неориентированный. Приём короткий, стоит запомнить.

Обратите внимание: считая количество дорог, мы проверяем `вес != 0`,
а не суммируем — сумма дала бы длину, а не количество.

## Пробуем сами

### Задача 1. Степень по матрице

По матрице смежности и номеру вершины (с нуля) верните её степень —
количество ненулевых элементов в строке.

In [ ]:
def степень(матрица, номер):
    return ...

In [ ]:
si.check("1", степень, [
    (([[0, 1, 1], [1, 0, 0], [1, 0, 0]], 0), 2),
    (([[0, 1, 1], [1, 0, 0], [1, 0, 0]], 1), 1),
    (([[0, 0], [0, 0]], 0), 0),
    (([[0, 4, 7], [4, 0, 2], [7, 2, 0]], 0), 2),
])

### Задача 2. Проверка симметричности

Проверьте, симметрична ли матрица относительно главной диагонали.
Верните `True` или `False`.

Условие симметричности: `матрица[i][j] == матрица[j][i]` для всех пар.

In [ ]:
def симметрична(матрица):
    return ...

In [ ]:
si.check("2", симметрична, [
    ([[0, 1], [1, 0]], True),
    ([[0, 1], [0, 0]], False),
    ([[0, 4, 7], [4, 0, 2], [7, 2, 0]], True),
    ([[0]], True),
])

### Задача 3. Есть ли ребро

По матрице и двум номерам вершин верните `True`, если между ними
есть связь.

In [ ]:
def есть_ребро(матрица, из_, в):
    return ...

In [ ]:
si.check("3", есть_ребро, [
    (([[0, 1, 0], [1, 0, 1], [0, 1, 0]], 0, 1), True),
    (([[0, 1, 0], [1, 0, 1], [0, 1, 0]], 0, 2), False),
    (([[0, 5], [5, 0]], 0, 1), True),
])

## Домашнее задание

### Домашнее задание 1. Вершина с наибольшей степенью

По матрице смежности и списку имён верните имя вершины с наибольшим
числом связей. Если таких несколько — верните первую по порядку.

In [ ]:
def самая_связная(матрица, имена):
    return ...

In [ ]:
si.check("дз1", самая_связная, [
    (([[0, 4, 7, 0, 0],
       [4, 0, 2, 9, 0],
       [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5],
       [0, 0, 6, 5, 0]], ["А", "Б", "В", "Г", "Д"]), "В"),
    (([[0, 1], [1, 0]], ["А", "Б"]), "А"),
    (([[0, 0, 0], [0, 0, 1], [0, 1, 0]], ["А", "Б", "В"]), "Б"),
])

### Домашнее задание 2. Из списка смежности в матрицу

Обратное преобразование к примеру 2: по списку смежности и порядку имён
постройте матрицу смежности из нулей и единиц.

Подсказка: сначала создайте матрицу из нулей нужного размера,
затем расставьте единицы.

Создать матрицу n×n из нулей можно так:
`[[0] * n for _ in range(n)]`

> ⚠️ Не пишите `[[0] * n] * n` — так получится n ссылок на **одну и ту же**
> строку, и изменение одной изменит все. Классическая ловушка.

In [ ]:
def в_матрицу(граф, имена):
    return ...

In [ ]:
si.check("дз2", в_матрицу, [
    (({"А": ["Б"], "Б": ["А"]}, ["А", "Б"]), [[0, 1], [1, 0]]),
    (({"А": ["Б", "В"], "Б": ["А"], "В": ["А"]}, ["А", "Б", "В"]),
     [[0, 1, 1], [1, 0, 0], [1, 0, 0]]),
    (({"А": [], "Б": []}, ["А", "Б"]), [[0, 0], [0, 0]]),
])

### Домашнее задание 3. Задача формата ОГЭ

> В таблице отражена протяжённость дорог между пунктами.
> Определите **суммарную длину всех дорог** в этой сети.

Внимание на подвох: матрица симметрична, поэтому каждая дорога
записана в ней **дважды**.

In [ ]:
def общая_длина(матрица):
    return ...

In [ ]:
si.check("дз3", общая_длина, [
    ([[0, 4, 7, 0, 0],
      [4, 0, 2, 9, 0],
      [7, 2, 0, 3, 6],
      [0, 9, 3, 0, 5],
      [0, 0, 6, 5, 0]], 36),
    ([[0, 5], [5, 0]], 5),
    ([[0, 0], [0, 0]], 0),
])

---

### Совет к экзамену

Получив таблицу дорог, **первым делом рисуйте граф**. Это занимает
полминуты и убирает почти все ошибки: по картинке видно, куда можно
пройти, а по таблице приходится каждый раз искать нужную клетку.

А на следующем уроке мы научимся находить по такому графу
кратчайший путь — самое частое задание этого типа.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 4](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-04.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 6 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-06.ipynb)